In [1]:
import pandas as pd
import numpy as np

In [223]:
affiliation_appearance = pd.read_csv('data/affiliation_appearance.csv')

character_df = pd.read_csv('data/character_info_v2.csv')
character_affiliations = pd.read_csv('data/character_affiliations.csv')

character_appearance = pd.read_csv('data/character_appearance.csv')

In [224]:
character_appearance['name'] = character_df['Official English Name']

In [225]:
character_appearance.head()

,name,1,2,3,4,5,6,7,8,9,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
0,A.O.,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Abdullah,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Absalom,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Agilia,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Adelle,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Feature Engineering

## Character Appearance Features

`recent_freq` - average appearance in last n=100 chaps

In [226]:
character_appearance_long = character_appearance.melt(
    id_vars=['name'],
    var_name='chapter',
    value_name='appeared'
)

character_appearance_long['rolling_appearance'] = (
    character_appearance_long.sort_values('chapter')
    .groupby('name')['appeared']
    .rolling(window=25).sum()
    .reset_index(0, drop=True)
)

In [127]:
character_appearance_long['chapter'] = character_appearance_long['chapter'].astype(int)

In [162]:
recent_cutoff = character_appearance_long['chapter'].max() - 100

recent_freq = (
    character_appearance_long[character_appearance_long['chapter'] > recent_cutoff]
    .groupby('name')['appeared']
    .mean()
    .reset_index(name='recent_freq')
)

total_app = (
    character_appearance_long.groupby('name')['appeared']
    .sum()
    .reset_index(name='total_appearances')
)

metrics = recent_freq.merge(total_app, on='name')

metrics['activity_score'] = (
    metrics['recent_freq'] * 0.7 +
    (metrics['total_appearances'] / metrics['total_appearances'].max()) * 0.3
)

In [167]:
metrics.sort_values(by='activity_score', ascending=False).head()

,name,recent_freq,total_appearances,activity_score
901,Monkey D. Luffy,0.56,902,0.692000
951,Nami,0.53,695,0.602153
1376,Usopp,0.51,594,0.554561
1173,Sanji,0.49,627,0.551537
1128,Roronoa Zoro,0.44,660,0.527512


In [240]:
metrics.to_csv('data/usable/relevance_metrics.csv')

In [186]:
time_series = character_appearance_long[character_appearance_long['appeared']==1]
time_series.to_csv('data/usable/time_series.csv')

In [188]:
rolling_chars = character_appearance_long.sort_values(['name', 'chapter'])

rolling_chars['rolling_appearance'] = (
    rolling_chars.groupby('name')['appeared']
    .rolling(window=20)
    .sum()
    .reset_index(level=0, drop=True)
)

In [189]:
rolling_chars.to_csv('data/usable/rolling_chars.csv')

# Affiliation Appearance Features

In [135]:
affiliation_appearance

,Affiliation,1,2,3,4,5,6,7,8,9,...,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176
0,Caesar Clown,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Wapol,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Pacifista,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Laboon,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Guardians,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312,G-1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
313,Tontatta Pirates,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
314,Gol D. Roger,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
315,Will of D.,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [227]:
affiliation_appearance_long = affiliation_appearance.melt(
    id_vars=['Affiliation'],
    var_name='chapter',
    value_name='appeared'
)

affiliation_appearance_long['rolling_appearance'] = (
    affiliation_appearance_long.sort_values('chapter')
    .groupby('Affiliation')['appeared']
    .rolling(window=25).sum()
    .reset_index(0, drop=True)
)

In [138]:
affiliation_appearance_long[affiliation_appearance_long['Affiliation']=='Straw Hat Pirates']

,Affiliation,chapter,appeared,rolling_appearance
301,Straw Hat Pirates,1,0,NaN
618,Straw Hat Pirates,2,0,23.0
935,Straw Hat Pirates,3,0,19.0
1252,Straw Hat Pirates,4,0,19.0
1569,Straw Hat Pirates,5,1,24.0
...,...,...,...,...
371508,Straw Hat Pirates,1172,1,13.0
371825,Straw Hat Pirates,1173,1,13.0
372142,Straw Hat Pirates,1174,1,13.0
372459,Straw Hat Pirates,1175,1,13.0


# Affiliation Dataframe

In [222]:
character_df[character_df['Official English Name']=="Olombus"]

,Unnamed: 0,Official English Name,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty
1078,1078,Olombus,704.0,"['Yonta Maria Grand Fleet', 'Straw Hat Grand F...",Captain of the Seventh Ship of the Straw Hat G...,Alive,June 23,0,0,0,0,Grand Line,42.0,480.0,X,NaN,1.480000e+16


In [ ]:
character_affiliations['Official English Name'] = character_df['Official English Name']

character_affiliations_long = character_affiliations.melt(
    id_vars=['Official English Name'],
    var_name='affiliation',
    value_name='member'
)

In [239]:
character_affiliations_long[character_affiliations_long['member']==1].to_csv('data/usable/character_affiliations_long.csv')

# Affiliation Descriptions

In [232]:
merged = character_affiliations_long[character_affiliations_long['member']==1].merge(character_df, on='Official English Name')

In [197]:
merged.head()

,Official English Name,affiliation,member,Unnamed: 0,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty
0,Browndros BeardriguezChadros Higelyges,Caesar Clown,1,190,581.0,"['Caesar Clown', 'Centaur Patrol Unit', 'Brown...",Leader of the Centaur Patrol Unit (former),Alive,February 3,0,0,0,0,Grand Line,43.0,691.0,S,NaN,80060000.0
1,Chappe,Caesar Clown,1,194,668.0,"['Caesar Clown', 'Centaur Patrol Unit']",NaN,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,Dragon Number Twenty-One,Caesar Clown,1,393,675.0,"['Caesar Clown', 'Vegapunk']",NaN,Unknown,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,Fen Bock,Caesar Clown,1,429,668.0,"['Caesar Clown', 'Centaur Patrol Unit']",NaN,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,Hyoutauros,Caesar Clown,1,622,658.0,"['Caesar Clown', 'Centaur Patrol Unit']",NaN,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [233]:
crew_stats = (
    merged.groupby('affiliation')
    .agg(
        total_bounty=('Bounty', 'sum'),
        avg_bounty=('Bounty', 'mean'),
        crew_size=('Official English Name', 'nunique'),
        avg_age=('Age','mean'),
        avg_height=('Height','mean')
    )
    .reset_index()
)

In [236]:
pirate_crews = crew_stats[crew_stats['affiliation'].str.contains(r'Pirates|Cross Guild', regex=True, na=False)]


In [268]:
pirate_crews['road_poneglyph'] = 0

In [275]:
pirate_crews[pirate_crews['affiliation']=='Red Hair Pirates']

,affiliation,total_bounty,avg_bounty,crew_size,avg_age,avg_height,road_poneglyph
232,Red Hair Pirates,4.142900e+09,828580000.0,17,38.4,186.2,0


In [276]:
pirate_crews.at[273, 'road_poneglyph'] = 3
pirate_crews.at[25, 'road_poneglyph'] = 2
pirate_crews.at[23, 'road_poneglyph'] = 1
pirate_crews.at[232, 'road_poneglyph'] = 2

In [277]:
pirate_crews.sort_values(by='road_poneglyph')

,affiliation,total_bounty,avg_bounty,crew_size,avg_age,avg_height,road_poneglyph
3,A O Pirates,0.000000e+00,NaN,1,NaN,NaN,0
4,Acumate Pirates,6.000000e+07,6.000000e+07,1,NaN,NaN,0
5,Alvida Pirates,5.050000e+08,2.525000e+08,5,20.500000,182.500000,0
11,Arlong Pirates,4.085000e+08,8.170000e+07,10,29.333333,793.666667,0
16,Barrels Pirates,2.220000e+08,2.220000e+08,2,31.000000,233.000000,0
...,...,...,...,...,...,...,...
306,Who's-Who Pirates,5.460000e+08,5.460000e+08,1,38.000000,336.000000,0
23,Big Mom Pirates,1.157150e+10,7.714333e+08,94,31.080000,454.043478,1
232,Red Hair Pirates,4.142900e+09,8.285800e+08,17,38.400000,186.200000,2
25,Blackbeard Pirates,4.246200e+09,3.860182e+08,17,41.000000,1981.636364,2


In [237]:
pirate_crews.to_csv('data/usable/crew_data.csv')

# Character Features

In [328]:
character_df[character_df['Official English Name']=='Monkey D. Luffy']

,Unnamed: 0,Official English Name,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty,norm_bounty,haki_score
962,962,Monkey D. Luffy,1.0,"['Straw Hat Pirates', 'Straw Hat Grand Fleet',...",Pirate Captain,Alive,May 5,1,1,1,1,East Blue,19.0,174.0,F,Paramecia (Mythical Zoan),3.000000e+09,0.972466,1.0


In [329]:
character_df['norm_bounty'] = np.log1p(character_df['Bounty']) / np.log1p(character_df['Bounty'].max())

In [411]:
character_df['haki_score'] = (
    0.1 * character_df['Observation Haki'] +
    0.3 * character_df['Armament Haki'] +
    0.6 * character_df['Conquerers Haki']
)

In [331]:
character_df[character_df['Official English Name']=='Monkey D. Luffy']

,Unnamed: 0,Official English Name,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty,norm_bounty,haki_score
962,962,Monkey D. Luffy,1.0,"['Straw Hat Pirates', 'Straw Hat Grand Fleet',...",Pirate Captain,Alive,May 5,1,1,1,1,East Blue,19.0,174.0,F,Paramecia (Mythical Zoan),3.000000e+09,0.972466,1.0


In [332]:
metrics.rename(columns={'name': 'Official English Name'}, inplace=True)

In [412]:
char_df = pd.merge(character_df, metrics, on='Official English Name', how='left')

In [413]:
active = ['Alive', 'Active']

char_df['alive'] = char_df['Status'].isin(active)

In [414]:
disbanded = ['Roger Pirates', 
             'Whitebeard Pirates', 
             'Rocks Pirates', 
             'Buggy Pirates', 
             'Usopp Pirates',
             'Bellamy Pirates',
             'Alvida Pirates',
             'Rumbar Pirates',
             'Donquixote Pirates',
             'Arlong Pirates',
             'Spade Pirates',
             'Gecko Pirates',
             'Sun Pirates',
             'Krieg Pirates',
             "Who's-Who Pirates",
             ]

pirate_chars = character_affiliations_long[
    (character_affiliations_long['affiliation'].str.contains(r'Pirates|Cross Guild', regex=True, na=False))
    & (character_affiliations_long['member']==1)].dropna().reset_index(drop=True)

pirate_chars = pirate_chars[~pirate_chars['affiliation'].isin(disbanded)]
# pirate_chars.sort_values(by='Official English Name').reset_index(drop=True).to_csv('data/pirate_chars.csv')

In [415]:
char_df = pd.merge(char_df, pirate_chars, on='Official English Name', how='left')

In [416]:
pirate_crews['norm_crew_bounty'] = np.log1p(pirate_crews['total_bounty']) / np.log1p(pirate_crews['total_bounty'].max())

In [417]:
pirate_crews['crew_score'] = (
    0.6 * pirate_crews['norm_crew_bounty']
    + 0.4 * (pirate_crews['road_poneglyph'] / 4)
)

In [418]:
char_df = pd.merge(char_df, pirate_crews, on='affiliation', how='left')

In [419]:
char_df['closeness'] = (
    0.3 * char_df['norm_bounty'] +
    0.2 * char_df['activity_score'] +
    0.3  * char_df['haki_score'] +
    0.2  * char_df['crew_score']
)

char_df.loc[char_df['alive'] == 0, 'closeness'] = 0

In [421]:
char_df_filter = char_df[
    ['Official English Name',
     'Birthday',
     'Devil Fruit',
     'Observation Haki',
     'Armament Haki',
     'Conquerers Haki',
     'Origin',
     'Age',
     'Height',
     'Blood Type',
     'Bounty',
     'norm_bounty',
     'haki_score',
     'recent_freq',
     'total_appearances',
     'activity_score',
     'alive',
     'affiliation',
     'total_bounty',
     'norm_crew_bounty',
     'road_poneglyph',
     'crew_score',
     'closeness']]

In [424]:
affil_closeness = (
    char_df_filter.groupby('affiliation')
    .agg(
        avg_closeness=('closeness', 'mean'),
        max_closeness=('closeness', 'max')
    )
    .reset_index()
)

# To use

In [431]:
char_df_filter.to_csv('final_data/characters.csv')

In [432]:
pirate_crews.to_csv('final_data/crews.csv')

In [433]:
affil_closeness.to_csv('final_data/affiliation_closeness.csv')

In [434]:
affiliation_appearance_long.to_csv('final_data/affiliations.csv')

# Network Graph

In [438]:
character_affiliations_long[character_affiliations_long['member']==1]

,Official English Name,affiliation,member
190,Browndros BeardriguezChadros Higelyges,Caesar Clown,1
194,Chappe,Caesar Clown,1
393,Dragon Number Twenty-One,Caesar Clown,1
429,Fen Bock,Caesar Clown,1
622,Hyoutauros,Caesar Clown,1
...,...,...,...
490957,Portgaz D. Rouge,Will of D.,1
491020,Rocks D. Xebec,Will of D.,1
491246,Trafalgar D. Water Law,Will of D.,1
492269,Mash,Sea Floor,1


In [464]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import defaultdict

name_col = "Official English Name"
affil_col = "affiliation"
member_col = "member"

# Optional: remove very large affiliations to reduce clutter
# Example: if an affiliation has more than 40 members, skip it
max_affiliation_size = None   # set to something like 40 if needed

# Layout reproducibility
layout_seed = 42

# =========================
# LOAD DATA
# =========================
df = character_affiliations_long[character_affiliations_long['member']==1]

# Drop blanks / duplicates
df = df.dropna(subset=[name_col, affil_col])
df = df.drop_duplicates(subset=[name_col, affil_col])

# =========================
# BUILD CHARACTER-CHARACTER EDGES
# =========================
# For every affiliation, connect every pair of members
edge_weights = defaultdict(int)
edge_affiliations = defaultdict(list)

grouped = df.groupby(affil_col)[name_col].apply(lambda s: sorted(set(s))).reset_index()

for _, row in grouped.iterrows():
    affiliation = row[affil_col]
    members = row[name_col]

    if max_affiliation_size is not None and len(members) > max_affiliation_size:
        continue

    # Make all unique character pairs within the affiliation
    for a, b in combinations(members, 2):
        edge = tuple(sorted((a, b)))
        edge_weights[edge] += 1
        edge_affiliations[edge].append(affiliation)

# =========================
# BUILD GRAPH
# =========================
G = nx.Graph()

# Add all characters as nodes
all_characters = sorted(df[name_col].unique())
G.add_nodes_from(all_characters)

# Add weighted edges
for (a, b), w in edge_weights.items():
    G.add_edge(
        a,
        b,
        weight=w,
        shared_affiliations=", ".join(edge_affiliations[(a, b)])
    )

# Remove isolated nodes if you only want connected characters
isolates = list(nx.isolates(G))
G.remove_nodes_from(isolates)

# =========================
# COMPUTE LAYOUT
# =========================
pos = nx.spring_layout(G, seed=layout_seed, k=None)

# =========================
# NODE METRICS
# =========================
degree_dict = dict(G.degree())
weighted_degree_dict = dict(G.degree(weight="weight"))

# =========================
# EXPORT TABLEAU-FRIENDLY ONE-FILE DATA
# =========================
# We create:
# 1) edge rows: two rows per edge so Tableau can draw lines using PathOrder
# 2) node rows: one row per character for circles / labels

edge_rows = []

# Edge rows
edge_id = 1
for source, target, attrs in G.edges(data=True):
    weight = attrs.get("weight", 1)
    shared_affiliations = attrs.get("shared_affiliations", "")

    x1, y1 = pos[source]
    x2, y2 = pos[target]

    edge_rows.append({
        "RowType": "edge",
        "EdgeID": edge_id,
        "PathOrder": 1,
        "Character": source,
        "ConnectedCharacter": target,
        "X": x1,
        "Y": y1,
        "Degree": degree_dict[source],
        "WeightedDegree": weighted_degree_dict[source],
        "EdgeWeight": weight,
        "SharedAffiliations": shared_affiliations
    })

    edge_rows.append({
        "RowType": "edge",
        "EdgeID": edge_id,
        "PathOrder": 2,
        "Character": target,
        "ConnectedCharacter": source,
        "X": x2,
        "Y": y2,
        "Degree": degree_dict[target],
        "WeightedDegree": weighted_degree_dict[target],
        "EdgeWeight": weight,
        "SharedAffiliations": shared_affiliations
    })

    edge_id += 1

node_rows = []

# Node rows
for node in G.nodes():
    x, y = pos[node]
    size = (degree_dict[node] ** 0.5) * 60
    node_rows.append({
        "RowType": "node",
        "EdgeID": None,
        "PathOrder": None,
        "Character": node,
        "ConnectedCharacter": None,
        'node_size': size,
        "X": x,
        "Y": y,
        "Degree": degree_dict[node],
        "WeightedDegree": weighted_degree_dict[node],
        "EdgeWeight": None,
        "SharedAffiliations": None
    })

out_df = pd.DataFrame(edge_rows)
node_df = pd.DataFrame(node_rows)
out_df.to_csv('final_data/network_edge.csv', index=False)
node_df.to_csv('final_data/network_node.csv', index=False)

# Present Affils


In [470]:
character_appearance_long.to_csv('final_data/char_appearance_long.csv')

In [466]:
present_affils = pd.merge(char_df, character_affiliations_long[character_affiliations_long['member']==1], on='affiliation', how='left')

In [468]:
present_affils[['Official English Name_x','affiliation']]

,Official English Name_x,affiliation
0,A.O.,A O Pirates
1,Abdullah,Ideo Pirates
2,Abdullah,Ideo Pirates
3,Abdullah,Ideo Pirates
4,Abdullah,Ideo Pirates
...,...,...
30901,Zodia,NaN
30902,Burr,NaN
30903,Zukka,NaN
30904,Zunesha,NaN


In [471]:
char_df_filter.head()

,Official English Name,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,...,recent_freq,total_appearances,activity_score,alive,affiliation,total_bounty,norm_crew_bounty,road_poneglyph,crew_score,closeness
0,A.O.,January 15,0,0,0,0,NaN,NaN,NaN,NaN,...,0.00,7.0,0.002328,False,A O Pirates,0.000000e+00,0.000000,0.0,0.000000,0.0
1,Abdullah,NaN,0,0,0,0,NaN,NaN,NaN,NaN,...,0.01,22.0,0.014317,True,Ideo Pirates,0.000000e+00,0.000000,0.0,0.000000,NaN
2,Absalom,December 30,1,0,0,0,West Blue,34.0,195.0,F,...,0.00,17.0,0.005654,False,Thriller Bark Pirates,1.290000e+09,0.895255,0.0,0.537153,0.0
3,Agilia,NaN,0,0,0,0,NaN,NaN,NaN,NaN,...,0.00,2.0,0.000665,True,NaN,NaN,NaN,NaN,NaN,NaN
4,Adelle,NaN,0,0,0,0,Grand Line,NaN,NaN,NaN,...,0.00,2.0,0.000665,True,NaN,NaN,NaN,NaN,NaN,NaN
